# Exercise 04 — HTTP Client

Everything in AI Engineering goes over HTTP: the OpenAI API, your vector database, your observability tools, your own FastAPI service. If you're not comfortable with how HTTP requests work, you'll spend a lot of time confused about errors that have nothing to do with AI.

This exercise uses the [Open-Meteo](https://open-meteo.com) weather API. It's free, has no key, and returns real data. We'll fetch current weather for a few cities and parse the response.

**What you'll practice:**
- Making GET requests with `httpx`
- Reading query parameters, status codes, and response JSON
- Handling errors gracefully

In [ ]:
!pip install httpx -q

In [ ]:
import httpx

# Open-Meteo: free weather API, no key needed
BASE_URL = "https://api.open-meteo.com/v1/forecast"

# Coordinates for a few cities
CITIES = {
    "New York":    {"latitude": 40.71, "longitude": -74.01},
    "London":      {"latitude": 51.51, "longitude": -0.13},
    "Tokyo":       {"latitude": 35.68, "longitude": 139.69},
}


def get_current_temperature(city: str, latitude: float, longitude: float) -> dict:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,wind_speed_10m",
        "temperature_unit": "celsius",
    }

    with httpx.Client(timeout=10.0) as client:
        response = client.get(BASE_URL, params=params)
        response.raise_for_status()

    data = response.json()
    current = data["current"]

    return {
        "city": city,
        "temperature_c": current["temperature_2m"],
        "wind_speed_kmh": current["wind_speed_10m"],
    }


for city, coords in CITIES.items():
    result = get_current_temperature(city, **coords)
    print(f"{result['city']}: {result['temperature_c']}°C, wind {result['wind_speed_kmh']} km/h")

## Inspecting the raw response

Before you parse anything, it helps to see the full response. This is something you'll do constantly when working with new APIs.

In [ ]:
import json

params = {
    "latitude": 40.71,
    "longitude": -74.01,
    "current": "temperature_2m,wind_speed_10m",
}

with httpx.Client() as client:
    response = client.get(BASE_URL, params=params)

print(f"Status: {response.status_code}")
print(f"URL called: {response.url}")
print()
print(json.dumps(response.json(), indent=2))

## Error handling

Things break. The API is down, the network times out, you pass a bad parameter. This is how you handle it without your whole program crashing.

In [ ]:
def safe_get_temperature(city: str, latitude: float, longitude: float) -> dict | None:
    try:
        return get_current_temperature(city, latitude, longitude)
    except httpx.TimeoutException:
        print(f"{city}: request timed out")
    except httpx.HTTPStatusError as e:
        print(f"{city}: API error {e.response.status_code}")
    except httpx.RequestError as e:
        print(f"{city}: connection failed — {e}")
    return None


# Trigger a 400 with bad coordinates to see what the error looks like
bad = safe_get_temperature("Nowhere", latitude=999, longitude=999)
print(f"Result: {bad}")

## Check your understanding

- What does `raise_for_status()` do with a 200 response? What about a 429?
- Why use `httpx` over the built-in `urllib`? When would the built-in be enough?
- Modify `get_current_temperature` to also return humidity. Check the Open-Meteo docs to see what parameter name to add.